## MosKita — YOLOv8 Training
### Dengue Mosquito Breeding Site Detection

This notebook trains a YOLOv8s object detection model to identify *Aedes aegypti* / *Aedes albopictus* breeding containers.
The dataset is assembled in YOLOv8 format with **8 container classes (V1, bottle removed)** after removing broad context labels such as `breeding_place` and `stagnant_puddle`.

**Hardware:** Lenovo Legion 5 — RTX 2060 6 GB, Ryzen 7 4800H, 16 GB RAM

**Workflow:**
1. Environment & GPU check
2. Configuration ← *edit here before running*
3. Dataset assembly (merge selected sources)
4. Dataset verification & visualisation
5. Model training
6. Results & metrics
7. Model export (ONNX / TFLite)
8. Quick inference test
9. Summary & next steps


---
### 1 · Environment Setup

In [ ]:
import os
import sys
import time
import random
import yaml
import glob

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from collections import Counter

# Attempt to import ultralytics and YOLO; install if missing
try:
    from ultralytics import YOLO
    import ultralytics
except ModuleNotFoundError:
    print("\u26a0\ufe0f  'ultralytics' package not found. Installing via pip...")
    import subprocess
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics"])
    except Exception as e:
        print(f"\u26a0\ufe0f  Failed to install ultralytics: {e}")
        raise
    # Retry import after installation
    from ultralytics import YOLO
    import ultralytics

print("\u2705 All libraries imported successfully!")
print(f"   PyTorch version      : {torch.__version__}")
print(f"   Ultralytics version  : {ultralytics.__version__}")
print(f"   CUDA available       : {torch.cuda.is_available()}")
print(f"   Device count         : {torch.cuda.device_count()}")
print(f"   Python               : {sys.version.split()[0]}")


#### Detect GPU Available, Details, CUDA, and cuDNN

In [ ]:
print("=" * 50)
print("\U0001f5a5\ufe0f  GPU INFORMATION")
print("=" * 50)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    gpu_cap  = torch.cuda.get_device_capability(0)
    print(f"\u2705 GPU Detected         : {gpu_name}")
    print(f"   \u2022 Compute Capability : {gpu_cap[0]}.{gpu_cap[1]}")
    print(f"   \u2022 Total VRAM         : {gpu_mem:.2f} GB")
    print(f"   \u2022 VRAM Allocated     : {torch.cuda.memory_allocated(0) / (1024**3):.2f} GB")
    print(f"   \u2022 VRAM Reserved      : {torch.cuda.memory_reserved(0) / (1024**3):.2f} GB")
    DEVICE = 0
    # With fixed input size (640×640), benchmark mode selects the fastest cuDNN kernels
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False  # determinism handled via SEED in config
else:
    print("\u26a0\ufe0f  No GPU detected — training will use CPU (very slow)")
    DEVICE = "cpu"

print("=" * 50)
print()
print("=" * 50)
print("\u26a1 CUDA / PYTORCH INFORMATION")
print("=" * 50)
print(f"\u2705 CUDA Available       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   \u2022 PyTorch CUDA Ver.  : {torch.version.cuda}")
    print(f"   \u2022 cuDNN Benchmark    : {torch.backends.cudnn.benchmark}")
print(f"   \u2022 PyTorch Version    : {torch.__version__}")
print(f"\u2705 cuDNN Version        : {torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else 'N/A'}")
print("=" * 50)


---
### 2 · Configuration

Edit the cell below — dataset sources, model variant, and all hyperparameters — then run it before proceeding.


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start=None):
    start_path = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start_path, *start_path.parents):
        if (candidate / "scripts" / "class_maps" / "v1_target_names.txt").exists():
            return candidate
    raise FileNotFoundError("Could not locate the MosKita repo root from the current working directory.")


REPO_ROOT = find_repo_root()
NOTEBOOK_DIR = REPO_ROOT / "notebooks"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.moskita_dataset_report import load_target_class_names

BANNED_CLASS_NAMES = {"bottle"}


def _normalize_class_name(name):
    return str(name).strip().lower().replace(" ", "_")


# ── DATASET SOURCE SELECTION ─────────────────────────────────
# Set enabled=True / False to include or exclude a source.
# After changing selections, re-run Section 3 (Dataset Assembly)
# to rebuild the training directory before training.
#
# Notes:
#   • adnans_breeding — mixed box/polygon export; polygon rows auto-converted.
#     Bottle rows are dropped by --drop-unmapped.
#   • adnans_water  — polygon segmentation; auto-converted to boxes.
#   • faiyaz_mosquitofusion — mosquito/swarm classes already removed;
#     only "Breeding Place" (→ breeding_place) remains.
#     Polygon rows are auto-converted to boxes.
#   • k1taru_local — self-curated Roboflow export with polygon labels;
#     polygons are auto-converted to boxes. Train split already has
#     4× photometric augmentation baked in.

DATASET_SOURCES = {
    "adnans_breeding": {
        "enabled":    True,
        "input_root": "../data/annotated/outsource/adnans/Breeding Place Detection",
        "class_map":  "../scripts/class_maps/adnans_breeding_to_v1.json",
        "split_map":  "train:train,valid:val,test:test",
        "extra_args": ["--drop-unmapped", "--convert-segments-to-boxes"],
        "note":       "Adnans — tires, drain inlets, vases, coconut shells (Bottle dropped)",
    },
    "adnans_water": {
        "enabled":    False,
        "input_root": "../data/annotated/outsource/adnans/Water Surface Segmentation",
        "class_map":  "../scripts/class_maps/adnans_water_to_v1.json",
        "split_map":  "train:train,valid:val,test:test",
        "extra_args": ["--convert-segments-to-boxes"],
        "note":       "Adnans — water surface segmentation (polygon → box conversion)",
    },
    "faiyaz_mosquitofusion": {
        "enabled":    True,
        "input_root": "../data/annotated/outsource/faiyazabdullah/MosquitoFusion Dataset",
        "class_map":  "../scripts/class_maps/faiyaz_mosquitofusion_to_v1.json",
        "split_map":  "train:train,valid:val,test:test",
        "extra_args": ["--drop-unmapped", "--convert-segments-to-boxes"],
        "note":       "Faiyaz — MosquitoFusion (Breeding Place → breeding_place)",
    },
    "roboflow_public": {
        "enabled":    True,
        "input_root": "../data/annotated/outsource/roboflow",
        "class_map":  "../scripts/class_maps/roboflow_to_v1.json",
        "split_map":  "train:train,valid:val,test:test",
        "extra_args": ["--drop-unmapped"],
        "note":       "Roboflow public examples (bucket, puddle, tire)",
    },
    "k1taru_local": {
        "enabled":    True,
        "input_root": "../data/annotated/k1taru",
        "class_map":  "../scripts/class_maps/k1taru_to_v1.json",
        "split_map":  "train:train,valid:val,test:test",
        "extra_args": ["--convert-segments-to-boxes"],
        "note":       "K1taru self-curated local dataset export",
    },
}

TARGET_NAMES_FILE = REPO_ROOT / "scripts" / "class_maps" / "v1_target_names.txt"
TARGET_CLASS_NAMES = load_target_class_names(TARGET_NAMES_FILE)
if not TARGET_CLASS_NAMES:
    raise ValueError(f"No class names could be parsed from: {TARGET_NAMES_FILE}")

normalized_target_names = {_normalize_class_name(name) for name in TARGET_CLASS_NAMES}
blocked_in_targets = sorted(BANNED_CLASS_NAMES.intersection(normalized_target_names))
if blocked_in_targets:
    raise ValueError(
        f"Banned classes are still present in target taxonomy: {blocked_in_targets}. "
        f"Update {TARGET_NAMES_FILE} before training."
    )

TARGET_NAMES = ",".join(TARGET_CLASS_NAMES)
NUM_CLASSES = len(TARGET_CLASS_NAMES)
CLASS_NAMES = TARGET_CLASS_NAMES

ASSEMBLED_DIR = str((REPO_ROOT / "data" / "annotated").resolve())
DATA_YAML = str((Path(ASSEMBLED_DIR) / "data.yaml").resolve())
PROJECT_DIR = str((REPO_ROOT / "models" / "runs").resolve())
LEGACY_PROJECT_DIR = str((NOTEBOOK_DIR / "runs" / "models" / "runs").resolve())
EXPORT_DIR = str((REPO_ROOT / "models" / "exports").resolve())

MODEL_VARIANT = "yolo26n.pt"
RUN_NAME = "moskita_v12"

IMG_SIZE = 640
EPOCHS = 2
BATCH_SIZE = 32
PATIENCE = 15
WORKERS = 7
SEED = 42
OPTIMIZER = "AdamW"
LR0 = 0.001
LRF = 0.01
WEIGHT_DECAY = 5e-4
WARMUP_EPOCHS = 3.0
WARMUP_MOMENTUM = 0.8
CACHE = False
EXIST_OK = False
SAVE_PERIOD = -1
COS_LR = True

HSV_H = 0.015
HSV_S = 0.7
HSV_V = 0.4
DEGREES = 10.0
TRANSLATE = 0.1
SCALE = 0.5
SHEAR = 2.0
PERSPECTIVE = 0.0
FLIPUD = 0.0
FLIPLR = 0.5
MOSAIC = 1.0
MIXUP = 0.0
COPY_PASTE = 0.0
CLOSE_MOSAIC = 10

CONF_THRESHOLD = 0.25
IOU_THRESHOLD = 0.50

SAFE_NOTEBOOK_MODE = True
LIVE_TRAIN_PROGRESS = True
LIVE_PROGRESS_REFRESH_SECONDS = 1.0
LIVE_PROGRESS_EVERY_N_BATCHES = 1
LIVE_PROGRESS_RECENT_EPOCHS = 6
TRAIN_PLOTS = not SAFE_NOTEBOOK_MODE
RUN_POST_TRAIN_VALIDATION = False
SHOW_DETAILED_EVAL_PLOTS = False
SHOW_DETAILED_INFERENCE_PLOTS = False
SHOW_TRAINING_ARTIFACT_GALLERY = False
MAX_NOTEBOOK_IMAGE_PANELS = 6

print("✅ Configuration loaded")
print(f"   Repo root         : {REPO_ROOT}")
print(f"   Target classes    : {NUM_CLASSES}")
print(f"   Class order       : {CLASS_NAMES}")
print(f"   Banned classes    : {sorted(BANNED_CLASS_NAMES)}")
print(f"   Data YAML         : {DATA_YAML}")
print(f"   Project directory : {PROJECT_DIR}")
print(f"   Model             : {MODEL_VARIANT}")
print(f"   Run name          : {RUN_NAME}")
print(f"   Epochs            : {EPOCHS}")
print(f"   Batch size        : {BATCH_SIZE}")
print(f"   Safe notebook     : {SAFE_NOTEBOOK_MODE}")
print(
    f"   Live progress     : {LIVE_TRAIN_PROGRESS} "
    f"(refresh={LIVE_PROGRESS_REFRESH_SECONDS:.1f}s, every={LIVE_PROGRESS_EVERY_N_BATCHES} batch)"
)
print(f"   Train plots       : {TRAIN_PLOTS}")
print(f"   Extra post-val    : {RUN_POST_TRAIN_VALIDATION}")
print(f"   Detailed visuals  : {SHOW_DETAILED_EVAL_PLOTS}")
print(f"   Max image panels  : {MAX_NOTEBOOK_IMAGE_PANELS}")
print()
print("Enabled sources:")
for src_name, src_cfg in DATASET_SOURCES.items():
    if src_cfg["enabled"]:
        print(f"   • {src_name:<22} — {src_cfg['note']}")


---
### 3 · Dataset Assembly

Toggle `enabled` flags in the **Configuration** cell, then run the cell below to clear and rebuild the training directory from the selected sources. Re-run any time the source selection changes.

> ⚠️ This overwrites `data/annotated/train|val|test/`. Source datasets must have class IDs compatible with V1 (handled automatically by the remap script + class maps in `scripts/class_maps/`).


In [ ]:
# Remap and merge selected sources into ASSEMBLED_DIR.
# Calls scripts/remap_yolo_dataset.py for each enabled source.
# ⚠️  This CLEARS data/annotated/train|val|test/ before rebuilding.

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import yaml

_nb_dir    = NOTEBOOK_DIR
_root_dir  = REPO_ROOT
_remap_py  = _root_dir / "scripts" / "remap_yolo_dataset.py"
_assembled = Path(ASSEMBLED_DIR).resolve()
_SPLITS    = ["train", "val", "test"]


def _ordered_names(names_value):
    if isinstance(names_value, list):
        return [str(name) for name in names_value]
    if isinstance(names_value, dict):
        def _sort_key(value):
            value_str = str(value)
            return int(value_str) if value_str.isdigit() else value_str

        return [str(names_value[key]) for key in sorted(names_value, key=_sort_key)]
    return []


# ── Clear existing splits ──────────────────────────────────────
print("🗑️  Clearing assembled dataset directories...")
for split in _SPLITS:
    for sub in ("images", "labels"):
        d = _assembled / split / sub
        if d.exists():
            shutil.rmtree(d)
        d.mkdir(parents=True, exist_ok=True)

# Clear the assembled YAML too so target-name renames rebuild cleanly.
_output_yaml = _assembled / "data.yaml"
if _output_yaml.exists():
    _output_yaml.unlink()
print()

# ── Process each enabled source ───────────────────────────────
print("📦 Assembling enabled sources:")
print("=" * 65)

_enabled_srcs = {k: v for k, v in DATASET_SOURCES.items() if v["enabled"]}

if not _enabled_srcs:
    print("⚠️  No sources are enabled — set enabled=True in the Configuration cell.")
else:
    for src_name, src_cfg in _enabled_srcs.items():
        _src_root = (_nb_dir / src_cfg["input_root"]).resolve()
        _map_file = (_nb_dir / src_cfg["class_map"]).resolve()

        print(f"▶  {src_name}")
        print(f"   {src_cfg['note']}")

        if not _src_root.exists():
            print(f"   ❌ INPUT NOT FOUND: {_src_root}")
            print()
            continue
        if not _map_file.exists():
            print(f"   ❌ CLASS MAP NOT FOUND: {_map_file}")
            print()
            continue

        _map_payload = json.loads(_map_file.read_text(encoding="utf-8"))
        _mapped_targets = {_normalize_class_name(name) for name in _map_payload.values()}
        _blocked_targets = sorted(BANNED_CLASS_NAMES.intersection(_mapped_targets))
        if _blocked_targets:
            raise ValueError(
                f"Source '{src_name}' maps to banned targets {_blocked_targets}: {_map_file}"
            )

        _cmd = [
            sys.executable, str(_remap_py),
            "--input-root",   str(_src_root),
            "--output-root",  str(_assembled),
            "--class-map",    str(_map_file),
            "--target-names", TARGET_NAMES,
            "--dataset-tag",  src_name,
            "--split-map",    src_cfg["split_map"],
        ] + src_cfg.get("extra_args", [])

        _proc = subprocess.run(_cmd, capture_output=True, text=True, cwd=str(_root_dir))

        if _proc.returncode != 0:
            print(f"   ❌ FAILED (exit {_proc.returncode})")
            for _line in (_proc.stderr or "").strip().splitlines()[-8:]:
                print(f"      {_line}")
        else:
            for _line in (_proc.stdout or "").strip().splitlines():
                if _line.strip():
                    print(f"   {_line.strip()}")
            print(f"   ✅ OK")
        print()

print("=" * 65)
print("Assembled totals:")
for split in _SPLITS:
    _n = len(list((_assembled / split / "images").glob("*")))
    print(f"   {split:6s} → {_n:5d} images")

if _output_yaml.exists():
    _assembled_cfg = yaml.safe_load(_output_yaml.read_text(encoding="utf-8")) or {}
    _assembled_names = _ordered_names(_assembled_cfg.get("names", {}))
    _normalized_names = {_normalize_class_name(name) for name in _assembled_names}
    _blocked_output = sorted(BANNED_CLASS_NAMES.intersection(_normalized_names))
    if _blocked_output:
        raise ValueError(
            f"Assembled dataset still contains banned classes: {_blocked_output}. "
            "Fix class maps and re-run this cell."
        )

print()
print("✅ Assembly complete — proceed to Dataset Verification ↓")


---
### 4 · Dataset Verification

Verify the assembled dataset structure, class counts, and annotation quality before training.


In [ ]:
# Load and display data.yaml
print("🔍 Loading dataset configuration...")
print()

with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

NUM_CLASSES = data_cfg["nc"]
CLASS_NAMES = [data_cfg["names"][i] for i in range(NUM_CLASSES)]
if CLASS_NAMES != TARGET_CLASS_NAMES:
    raise ValueError(
        "data.yaml class names do not match the configuration cell. "
        "Re-run Section 3 (Dataset Assembly) before training.\n"
        f"Expected: {TARGET_CLASS_NAMES}\nFound:    {CLASS_NAMES}"
    )

normalized_class_names = [_normalize_class_name(name) for name in CLASS_NAMES]
blocked_classes = sorted(BANNED_CLASS_NAMES.intersection(normalized_class_names))
if blocked_classes:
    raise ValueError(
        f"Banned classes are still present in data.yaml names: {blocked_classes}. "
        "Re-run Section 3 after fixing class maps."
    )

ALLOWED_CLASS_NAMES = [
    name for name in CLASS_NAMES if _normalize_class_name(name) not in BANNED_CLASS_NAMES
]

print(f"✅ Dataset config loaded: {DATA_YAML}")
print(f"   Number of classes : {NUM_CLASSES}")
print(f"   Classes:")
for i, name in enumerate(CLASS_NAMES):
    print(f"     {i}: {name}")
print(f"   Banned classes    : {sorted(BANNED_CLASS_NAMES)}")
print()
print(f"   Train images : {data_cfg.get('train', 'N/A')}")
print(f"   Val images   : {data_cfg.get('val', 'N/A')}")
print(f"   Test images  : {data_cfg.get('test', 'N/A')}")

In [ ]:
# Verify dataset split counts & annotation integrity
print("📊 DATASET SUMMARY")
print("=" * 70)

# Resolve the dataset root from data.yaml's 'path' field.
# Convention: 'path' is relative to the YAML file location.
_yaml_path_val = data_cfg.get("path", ".")
if os.path.isabs(_yaml_path_val):
    dataset_root = Path(_yaml_path_val)
else:
    _yaml_dir = Path(DATA_YAML).resolve().parent
    dataset_root = (_yaml_dir / _yaml_path_val).resolve()
dataset_root = str(dataset_root)

splits = ["train", "val", "test"]
split_stats = {}

for split in splits:
    img_dir   = os.path.join(dataset_root, split, "images")
    label_dir = os.path.join(dataset_root, split, "labels")

    if os.path.exists(img_dir):
        images = [f for f in os.listdir(img_dir)
                  if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))]
    else:
        images = []

    if os.path.exists(label_dir):
        labels = [f for f in os.listdir(label_dir) if f.endswith(".txt")]
    else:
        labels = []

    split_stats[split] = {"images": len(images), "labels": len(labels)}
    status = "✅" if len(images) > 0 and len(images) == len(labels) else "⚠️"
    print(f"   {status} {split:6s} — {len(images):5d} images, {len(labels):5d} labels")

total_images = sum(s["images"] for s in split_stats.values())
total_labels = sum(s["labels"] for s in split_stats.values())

print("=" * 70)
print(f"   Total: {total_images} images, {total_labels} labels")
print(f"   Dataset root: {dataset_root}")

if total_images == 0:
    print()
    print("⚠️  No images found! Please assemble the selected sources into data/annotated/")
    print("   Expected structure:")
    print("     data/annotated/train/images/  &  data/annotated/train/labels/")
    print("     data/annotated/val/images/    &  data/annotated/val/labels/")
    print("     data/annotated/test/images/   &  data/annotated/test/labels/")


In [ ]:
# Per-class annotation distribution (across all splits)
print("\U0001f4ca PER-CLASS ANNOTATION DISTRIBUTION")
print("=" * 70)

class_counts = Counter()

for split in splits:
    label_dir = os.path.join(dataset_root, split, "labels")
    if not os.path.exists(label_dir):
        continue
    for label_file in os.listdir(label_dir):
        if not label_file.endswith(".txt"):
            continue
        filepath = os.path.join(label_dir, label_file)
        with open(filepath, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:  # class_id + 4 bbox coords minimum
                    cls_id = int(parts[0])
                    class_counts[cls_id] += 1

if class_counts:
    print(f"{'Class ID':<10} {'Class Name':<30} {'Annotations':>12}")
    print("-" * 55)
    for i in range(NUM_CLASSES):
        count = class_counts.get(i, 0)
        print(f"{i:<10} {CLASS_NAMES[i]:<30} {count:>12}")
    print("-" * 55)
    print(f"{'Total':<41} {sum(class_counts.values()):>12}")
else:
    print("   No annotations found — dataset is empty or not yet exported.")

In [ ]:
# Visualize class distribution (bar chart)
if class_counts:
    fig, ax = plt.subplots(figsize=(12, 5))

    counts = [class_counts.get(i, 0) for i in range(NUM_CLASSES)]
    colors = plt.cm.Set3(np.linspace(0, 1, NUM_CLASSES))

    bars = ax.bar(CLASS_NAMES, counts, color=colors, edgecolor="black", linewidth=0.5)

    # Add count labels on bars
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                str(count), ha="center", va="bottom", fontsize=9, fontweight="bold")

    ax.set_title("MosKita — Per-Class Annotation Distribution", fontsize=14, fontweight="bold")
    ax.set_xlabel("Class", fontsize=11)
    ax.set_ylabel("Number of Annotations", fontsize=11)
    plt.xticks(rotation=45, ha="right", fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("\u26a0\ufe0f  Skipping visualization — no annotations found.")

In [ ]:

from utils.moskita_dataset_report import (
    plot_detection_dataset_overview,
    print_detection_dataset_summary,
    summarize_yolo_detection_dataset,
)

dataset_summary = summarize_yolo_detection_dataset(
    dataset_root,
    CLASS_NAMES,
    splits=splits,
)

print_detection_dataset_summary(dataset_summary)
plot_detection_dataset_overview(dataset_summary)

print("\n⚖️  Overall balance (all splits combined):")
for row in dataset_summary["balance_rows"]:
    flag = "  ⚠️  under-represented" if row["share_pct"] < 5 else ""
    bar = "█" * int(row["share_pct"] / 2)
    print(f"   {row['class_name']:<28} {row['share_pct']:5.1f}%  {bar}{flag}")


In [ ]:

# Bounding box size, aspect ratio, and annotation density analysis
print("📐 BOUNDING BOX & ANNOTATION ANALYSIS")
print("=" * 70)

all_widths, all_heights, all_areas, all_ratios, imgs_ann_count = [], [], [], [], []

for split in splits:
    lbl_dir = os.path.join(dataset_root, split, "labels")
    if not os.path.exists(lbl_dir):
        continue
    for lf in os.listdir(lbl_dir):
        if not lf.endswith(".txt"):
            continue
        rows = []
        with open(os.path.join(lbl_dir, lf)) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    rows.append(parts)
        imgs_ann_count.append(len(rows))
        for parts in rows:
            w, h = float(parts[3]), float(parts[4])
            all_widths.append(w)
            all_heights.append(h)
            all_areas.append(w * h)
            all_ratios.append(w / h if h > 0 else 0)

if all_areas:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # 1 — Box area distribution
    ax = axes[0, 0]
    ax.hist(all_areas, bins=60, color="#4C72B0", edgecolor="white", linewidth=0.4)
    ax.axvline(float(np.median(all_areas)), color="red",    linestyle="--",
               label=f"Median: {np.median(all_areas):.5f}")
    ax.axvline(float(np.mean(all_areas)),   color="orange", linestyle="--",
               label=f"Mean:   {np.mean(all_areas):.5f}")
    # COCO-style size thresholds (normalised area equivalents at 640px)
    for thresh, lbl in [(0.0032, "small|med"), (0.04, "med|large")]:
        ax.axvline(thresh, color="#AAAAAA", linestyle=":", linewidth=1.2)
        ax.text(thresh, ax.get_ylim()[1] * 0.92, lbl, fontsize=7, color="#888888", ha="left")
    ax.set_title("Bounding Box Area (normalised w×h)", fontweight="bold")
    ax.set_xlabel("Normalised area")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)

    # 2 — Aspect ratio distribution
    ax = axes[0, 1]
    clipped = np.clip(all_ratios, 0.05, 8)
    ax.hist(clipped, bins=60, color="#55A868", edgecolor="white", linewidth=0.4)
    ax.axvline(1.0, color="red",    linestyle="--", label="Square (1:1)")
    ax.axvline(float(np.median(clipped)), color="orange", linestyle="--",
               label=f"Median: {np.median(clipped):.2f}")
    ax.set_title("Bounding Box Aspect Ratio (w / h)", fontweight="bold")
    ax.set_xlabel("Width / Height")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)

    # 3 — Width vs Height scatter
    ax = axes[1, 0]
    _n = min(5000, len(all_widths))
    ax.scatter(all_widths[:_n], all_heights[:_n], alpha=0.18, s=7, c="#DD8452")
    ax.plot([0, 1], [0, 1], "r--", linewidth=1, label="Square diagonal")
    ax.set_title(f"Box Width vs Height (first {_n:,}, normalised)", fontweight="bold")
    ax.set_xlabel("Width (normalised)")
    ax.set_ylabel("Height (normalised)")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)

    # 4 — Annotations per image
    ax = axes[1, 1]
    _max_ann = max(imgs_ann_count)
    ax.hist(imgs_ann_count, bins=range(0, _max_ann + 2), color="#C44E52",
            edgecolor="white", linewidth=0.4)
    ax.axvline(float(np.mean(imgs_ann_count)), color="orange", linestyle="--",
               label=f"Mean: {np.mean(imgs_ann_count):.1f}")
    ax.set_title("Annotations per Image", fontweight="bold")
    ax.set_xlabel("Annotations")
    ax.set_ylabel("Images")
    ax.legend(fontsize=8)

    plt.suptitle("MosKita — Bounding Box & Annotation Analysis",
                 fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

    # Summary stats
    _total = len(all_areas)
    _small  = sum(1 for a in all_areas if a < 0.0032)
    _medium = sum(1 for a in all_areas if 0.0032 <= a < 0.04)
    _large  = sum(1 for a in all_areas if a >= 0.04)
    print(f"\n📏 Object size breakdown (COCO-style thresholds on normalised area):")
    print(f"   Small  (< ~32² px equiv.)  : {_small:5d}  ({_small  / _total * 100:.1f}%)")
    print(f"   Medium (~32²–128² px range): {_medium:5d}  ({_medium / _total * 100:.1f}%)")
    print(f"   Large  (> ~128² px equiv.) : {_large:5d}  ({_large  / _total * 100:.1f}%)")
    print(f"\n   Total annotations         : {_total:,}")
    print(f"   Median box area           : {np.median(all_areas):.5f}")
    print(f"   Mean aspect ratio (w/h)   : {np.mean(all_ratios):.3f}")
    print(f"   Mean annotations / image  : {np.mean(imgs_ann_count):.2f}  "
          f"(max: {max(imgs_ann_count)})")
else:
    print("⚠️  No annotations found — run Dataset Assembly first.")


In [ ]:

# Sample training images with ground-truth bounding boxes overlaid
import matplotlib.patches as mpatches


def _clean_sample_name(filename):
    name = os.path.basename(filename)
    stem, _ = os.path.splitext(name)
    if ".rf." in stem:
        stem = stem.split(".rf.", 1)[0]

    strip_prefixes = [
        "k1taru_local_train_",
        "k1taru_local_valid_",
        "k1taru_local_test_",
        "adnans_breeding_train_",
        "adnans_breeding_valid_",
        "adnans_breeding_test_",
        "roboflow_public_train_",
        "roboflow_public_valid_",
        "roboflow_public_test_",
        "k1taru_",
        "adnans_",
        "roboflow_",
    ]
    for prefix in strip_prefixes:
        if stem.startswith(prefix):
            stem = stem[len(prefix):]
            break

    return stem.replace("_", " ")


def _balanced_samples(items, total=8):
    total = min(total, len(items))
    if total <= 0:
        return []

    own_items = [item for item in items if os.path.basename(item).lower().startswith("k1taru_")]
    other_items = [item for item in items if item not in own_items]

    selected = []
    if own_items and other_items:
        own_quota = min(len(own_items), max(1, total // 3))
        other_quota = min(len(other_items), total - own_quota)
        selected.extend(random.sample(own_items, own_quota))
        selected.extend(random.sample(other_items, other_quota))
    else:
        selected.extend(random.sample(items, total))

    if len(selected) < total:
        chosen = set(selected)
        remaining = [item for item in items if item not in chosen]
        if remaining:
            selected.extend(random.sample(remaining, min(total - len(selected), len(remaining))))

    random.shuffle(selected)
    return selected


train_img_dir   = os.path.join(dataset_root, "train", "images")
train_label_dir = os.path.join(dataset_root, "train", "labels")

if os.path.exists(train_img_dir):
    _img_files = [f for f in os.listdir(train_img_dir)
                  if f.lower().endswith((".jpg", ".jpeg", ".png"))]

    if _img_files:
        selected = _balanced_samples(_img_files, total=8)
        num_samples = len(selected)
        _cmap = plt.get_cmap("tab10", NUM_CLASSES)

        cols = 4
        rows = (num_samples + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(16, 4.5 * rows))
        axes = axes.flatten() if num_samples > 1 else [axes]

        for idx, img_name in enumerate(selected):
            img_path = os.path.join(train_img_dir, img_name)
            lbl_path = os.path.join(train_label_dir,
                                    os.path.splitext(img_name)[0] + ".txt")
            img = mpimg.imread(img_path)
            ih, iw = img.shape[:2]
            axes[idx].imshow(img)

            if os.path.exists(lbl_path):
                with open(lbl_path) as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) < 5:
                            continue
                        cls_id = int(parts[0])
                        cx, cy, bw, bh = map(float, parts[1:5])
                        x1 = (cx - bw / 2) * iw
                        y1 = (cy - bh / 2) * ih
                        pw, ph = bw * iw, bh * ih
                        color = _cmap(cls_id % NUM_CLASSES)
                        rect = mpatches.FancyBboxPatch(
                            (x1, y1), pw, ph,
                            boxstyle="square,pad=0",
                            linewidth=1.5, edgecolor=color, facecolor="none",
                        )
                        axes[idx].add_patch(rect)
                        label = CLASS_NAMES[cls_id] if cls_id < NUM_CLASSES else str(cls_id)
                        axes[idx].text(
                            x1, max(y1 - 3, 0), label,
                            fontsize=6, color=color, fontweight="bold",
                            bbox=dict(boxstyle="square,pad=0.1",
                                      facecolor="black", alpha=0.55, linewidth=0),
                        )

            display_name = _clean_sample_name(img_name)
            axes[idx].set_title(display_name[:35], fontsize=7)
            axes[idx].axis("off")

        for idx in range(num_samples, len(axes)):
            axes[idx].axis("off")

        # Class legend
        _handles = [mpatches.Patch(color=_cmap(i), label=CLASS_NAMES[i])
                    for i in range(NUM_CLASSES)]
        fig.legend(_handles, [h.get_label() for h in _handles],
                   loc="lower center", ncol=4, fontsize=9,
                   title="Classes", bbox_to_anchor=(0.5, -0.02))
        plt.suptitle("Sample Training Images — Ground-Truth Boxes",
                     fontsize=14, fontweight="bold")
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️  No images in train/images/ — run Dataset Assembly first.")
else:
    print("⚠️  Train images directory not found.")


In [ ]:
# Image geometry and object placement overview
print("🖼️ IMAGE GEOMETRY & OBJECT PLACEMENT")
print("=" * 70)

try:
    from PIL import Image
except ModuleNotFoundError:
    Image = None
    print("⚠️  Pillow not available — skipping image geometry analysis.")

if Image is not None:
    image_records = []
    bbox_centers_x, bbox_centers_y = [], []
    bbox_areas_by_split = {split: [] for split in splits}
    valid_exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff")

    for split in splits:
        img_dir = os.path.join(dataset_root, split, "images")
        lbl_dir = os.path.join(dataset_root, split, "labels")
        if not os.path.exists(img_dir):
            continue

        for img_name in os.listdir(img_dir):
            if not img_name.lower().endswith(valid_exts):
                continue

            img_path = os.path.join(img_dir, img_name)
            try:
                with Image.open(img_path) as img_obj:
                    width, height = img_obj.size
            except Exception:
                continue

            aspect_ratio = width / height if height else np.nan
            image_records.append((split, width, height, aspect_ratio))

            lbl_path = os.path.join(lbl_dir, os.path.splitext(img_name)[0] + ".txt")
            if not os.path.exists(lbl_path):
                continue

            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cx, cy, bw, bh = map(float, parts[1:5])
                        bbox_centers_x.append(cx)
                        bbox_centers_y.append(cy)
                        bbox_areas_by_split[split].append(bw * bh)

    if image_records:
        split_palette = {"train": "#4C72B0", "val": "#DD8452", "test": "#55A868"}
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        ax = axes[0, 0]
        for split in splits:
            points = [(w, h) for rec_split, w, h, _ in image_records if rec_split == split]
            if not points:
                continue
            if len(points) > 1200:
                points = random.sample(points, 1200)
            widths, heights = zip(*points)
            ax.scatter(widths, heights, s=10, alpha=0.35, label=split,
                       color=split_palette.get(split, "#888888"))
        ax.set_title("Image Resolution Spread", fontweight="bold")
        ax.set_xlabel("Width (px)")
        ax.set_ylabel("Height (px)")
        ax.legend(fontsize=9)

        ax = axes[0, 1]
        for split in splits:
            ratios = [ratio for rec_split, _, _, ratio in image_records if rec_split == split and np.isfinite(ratio)]
            if ratios:
                ax.hist(ratios, bins=40, alpha=0.45, label=split, color=split_palette.get(split, "#888888"))
        ax.axvline(1.0, color="black", linestyle="--", linewidth=1, label="1:1")
        ax.set_title("Image Aspect Ratio Distribution", fontweight="bold")
        ax.set_xlabel("Width / Height")
        ax.set_ylabel("Images")
        ax.legend(fontsize=9)

        ax = axes[1, 0]
        if bbox_centers_x and bbox_centers_y:
            hb = ax.hexbin(bbox_centers_x, bbox_centers_y, gridsize=28, cmap="viridis", mincnt=1)
            fig.colorbar(hb, ax=ax, label="Boxes")
        ax.set_title("Bounding-Box Center Heatmap", fontweight="bold")
        ax.set_xlabel("Normalized x center")
        ax.set_ylabel("Normalized y center")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)

        ax = axes[1, 1]
        area_data = [bbox_areas_by_split[split] for split in splits if bbox_areas_by_split[split]]
        area_labels = [split for split in splits if bbox_areas_by_split[split]]
        if area_data:
            boxplot = ax.boxplot(area_data, labels=area_labels, patch_artist=True, showfliers=False)
            for patch, split in zip(boxplot["boxes"], area_labels):
                patch.set_facecolor(split_palette.get(split, "#888888"))
                patch.set_alpha(0.6)
            ax.set_yscale("log")
        ax.set_title("Box Area by Split", fontweight="bold")
        ax.set_ylabel("Normalized box area")

        plt.suptitle("MosKita — Pre-Training Data Coverage", fontsize=14, fontweight="bold")
        plt.tight_layout()
        plt.show()

        image_widths = [width for _, width, _, _ in image_records]
        image_heights = [height for _, _, height, _ in image_records]
        image_ratios = [ratio for _, _, _, ratio in image_records if np.isfinite(ratio)]
        print(f"   Images profiled          : {len(image_records):,}")
        print(f"   Boxes profiled           : {len(bbox_centers_x):,}")
        print(f"   Median image resolution  : {int(np.median(image_widths))} × {int(np.median(image_heights))}")
        print(f"   Median image ratio       : {np.median(image_ratios):.2f}")
    else:
        print("⚠️  No readable images found for geometry analysis.")

---
### 5 · Model Training

Fine-tune YOLOv8s on the assembled MosKita dataset. Weights, plots, and metrics are saved to `models/runs/`.


In [ ]:
# Load pretrained YOLOv8s
print("\U0001f680 Loading pretrained model...")
model = YOLO(MODEL_VARIANT)
print(f"\u2705 Model loaded: {MODEL_VARIANT}")
print(f"   Architecture : YOLOv8s (small)")
print(f"   Task         : Object Detection")

In [ ]:
# Train
import contextlib
import shutil
import tempfile
from pathlib import Path

from utils.moskita_run_report import (
    collect_saved_model_paths,
    format_duration,
    resolve_training_run_dir,
)
from utils.notebook_training import NotebookLiveTrainingProgress

print("🏋️  Starting training...")
print("=" * 50)
print("   Ultralytics console output will be redirected to train.log to keep VS Code stable.")
if LIVE_TRAIN_PROGRESS:
    print(
        "   Live notebook monitor enabled "
        f"(refresh={LIVE_PROGRESS_REFRESH_SECONDS:.1f}s, every={LIVE_PROGRESS_EVERY_N_BATCHES} batch)."
    )
else:
    print("   Live notebook monitor disabled. Set LIVE_TRAIN_PROGRESS = True to enable it.")

normalized_class_names = {_normalize_class_name(name) for name in CLASS_NAMES}
blocked_for_training = sorted(BANNED_CLASS_NAMES.intersection(normalized_class_names))
if blocked_for_training:
    raise ValueError(
        f"Training aborted: banned classes detected in CLASS_NAMES: {blocked_for_training}"
    )

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats(DEVICE)

NOTEBOOK_TRAIN_MONITOR = None
if LIVE_TRAIN_PROGRESS:
    NOTEBOOK_TRAIN_MONITOR = NotebookLiveTrainingProgress(
        run_name=RUN_NAME,
        model_variant=MODEL_VARIANT,
        refresh_seconds=LIVE_PROGRESS_REFRESH_SECONDS,
        every_n_batches=LIVE_PROGRESS_EVERY_N_BATCHES,
        recent_epoch_count=LIVE_PROGRESS_RECENT_EPOCHS,
    ).install(model)

_t_start = time.perf_counter()
_temp_train_log_path = None
with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", suffix=".train.log", delete=False) as _temp_train_log:
    _temp_train_log_path = Path(_temp_train_log.name)
    with contextlib.redirect_stdout(_temp_train_log), contextlib.redirect_stderr(_temp_train_log):
        results = model.train(
            data=DATA_YAML,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            name=RUN_NAME,
            project=PROJECT_DIR,
            exist_ok=EXIST_OK,
            patience=PATIENCE,
            optimizer=OPTIMIZER,
            lr0=LR0,
            lrf=LRF,
            cos_lr=COS_LR,
            weight_decay=WEIGHT_DECAY,
            warmup_epochs=WARMUP_EPOCHS,
            warmup_momentum=WARMUP_MOMENTUM,
            close_mosaic=CLOSE_MOSAIC,
            mosaic=MOSAIC,
            mixup=MIXUP,
            copy_paste=COPY_PASTE,
            flipud=FLIPUD,
            fliplr=FLIPLR,
            hsv_h=HSV_H,
            hsv_s=HSV_S,
            hsv_v=HSV_V,
            degrees=DEGREES,
            translate=TRANSLATE,
            scale=SCALE,
            shear=SHEAR,
            perspective=PERSPECTIVE,
            cache=CACHE,
            device=DEVICE,
            workers=WORKERS,
            seed=SEED,
            save_period=SAVE_PERIOD,
            plots=TRAIN_PLOTS,
            verbose=False,
        )

_elapsed = time.perf_counter() - _t_start
_preferred_run_dir = getattr(results, "save_dir", None) or getattr(model.trainer, "save_dir", None)
_run_dir = resolve_training_run_dir(
    run_name=RUN_NAME,
    project_dirs=[PROJECT_DIR, LEGACY_PROJECT_DIR],
    preferred_run_dir=_preferred_run_dir,
)
_saved_paths = collect_saved_model_paths(_run_dir)

TRAIN_RUN_DIR = _saved_paths["run_dir"]
TRAIN_RESULTS_CSV = _saved_paths["results_csv"]
TRAIN_BEST_WEIGHTS = _saved_paths["best_weights"]
TRAIN_LAST_WEIGHTS = _saved_paths["last_weights"]
TRAIN_LOG_PATH = _saved_paths["train_log"]
TRAIN_RUN_NAME = TRAIN_RUN_DIR.name
run_dir = TRAIN_RUN_DIR

TRAIN_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
if TRAIN_LOG_PATH.exists():
    TRAIN_LOG_PATH.unlink()
if _temp_train_log_path is not None and _temp_train_log_path.exists():
    shutil.move(str(_temp_train_log_path), str(TRAIN_LOG_PATH))

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print()
print("✅ Training complete!")
print(f"   Requested model : {MODEL_VARIANT}")
print(f"   Saved run name  : {TRAIN_RUN_NAME}")
print(f"   Run directory   : {TRAIN_RUN_DIR}")
print(f"   Training log    : {TRAIN_LOG_PATH}")
if TRAIN_BEST_WEIGHTS.exists():
    print(f"   Best weights    : {TRAIN_BEST_WEIGHTS.name}")
    print(f"   Best path       : {TRAIN_BEST_WEIGHTS}")
else:
    print("   Best weights    : not found")
if TRAIN_LAST_WEIGHTS.exists():
    print(f"   Last weights    : {TRAIN_LAST_WEIGHTS.name}")
    print(f"   Last path       : {TRAIN_LAST_WEIGHTS}")
else:
    print("   Last weights    : not found")
print(f"   Scheduler       : {'cosine' if COS_LR else 'linear'} decay")
print(f"   Train plots     : {TRAIN_PLOTS}")
print(f"   Total wall time : {format_duration(_elapsed)}")

if TRAIN_RESULTS_CSV.exists():
    import pandas as pd

    _train_df = pd.read_csv(TRAIN_RESULTS_CSV)
    _train_df.columns = _train_df.columns.str.strip()

    if "time" in _train_df.columns:
        _epoch_numbers = _train_df["epoch"].astype(int)
        _display_epochs = _epoch_numbers + 1 if _epoch_numbers.min() == 0 else _epoch_numbers
        _epoch_times = _train_df["time"].diff().fillna(_train_df["time"]).astype(float)
        print(f"   Epoch total     : {format_duration(_train_df['time'].iloc[-1])}")
        print(f"   Avg epoch time  : {format_duration(_epoch_times.mean())}")

        if "metrics/mAP50(B)" in _train_df.columns:
            _best_idx = _train_df["metrics/mAP50(B)"].idxmax()
            _best_epoch = int(_display_epochs.iloc[_best_idx])
            _best_epoch_time = float(_epoch_times.iloc[_best_idx])
            _best_map50 = float(_train_df.loc[_best_idx, "metrics/mAP50(B)"])
            print(f"   Best epoch      : {_best_epoch}")
            print(f"   Best epoch time : {format_duration(_best_epoch_time)}")
            print(f"   Best mAP@50     : {_best_map50:.4f}")

        _timing_table = pd.DataFrame(
            {
                "epoch": _display_epochs,
                "epoch_seconds": _epoch_times.round(2),
                "epoch_time": _epoch_times.map(format_duration),
            }
        )
        print()
        print("⏱️  Epoch timing breakdown")
        print(_timing_table.to_string(index=False))
    else:
        print("   Timing summary unavailable: results.csv does not contain a time column.")
else:
    print("   Timing summary unavailable: results.csv not found.")
    _log_lines = TRAIN_LOG_PATH.read_text(encoding="utf-8").splitlines() if TRAIN_LOG_PATH.exists() else []
    if _log_lines:
        print("   Training log tail:")
        for _line in _log_lines[-10:]:
            print(f"      {_line}")


---
### 6 · Results & Metrics

Display training curves, loss components, per-class performance, confusion matrix, and PR curves.


In [ ]:
import os
from pathlib import Path

from utils.moskita_run_report import collect_eval_plot_dirs, resolve_training_run_dir

# Locate the training run output directory
required_vars = ("PROJECT_DIR", "LEGACY_PROJECT_DIR", "RUN_NAME")
missing_vars = [name for name in required_vars if name not in globals()]
if missing_vars:
    raise NameError(
        f"Missing required setup variables: {', '.join(missing_vars)}. "
        "Run the setup/configuration cells above first."
    )

preferred_run_dir = globals().get("TRAIN_RUN_DIR")
run_dir = resolve_training_run_dir(
    run_name=RUN_NAME,
    project_dirs=[PROJECT_DIR, LEGACY_PROJECT_DIR],
    preferred_run_dir=preferred_run_dir,
)

print(f"📂 Training run directory: {run_dir}")
print()

# List available output files
if run_dir.exists():
    for item in sorted(os.listdir(run_dir)):
        print(f"   {item}")
else:
    print("⚠️  Run directory not found — training may not have completed.")

EVAL_PLOT_DIRS = collect_eval_plot_dirs(
    run_dir,
    repo_root=globals().get("REPO_ROOT"),
    notebook_dir=globals().get("NOTEBOOK_DIR"),
    cwd=Path.cwd(),
)

print()
print("🔎 Plot search directories:")
for path in EVAL_PLOT_DIRS[:6]:
    print(f"   - {path}")
if len(EVAL_PLOT_DIRS) > 6:
    print(f"   ... (+{len(EVAL_PLOT_DIRS) - 6} more)")


In [ ]:
# Display training results summary
import os
from pathlib import Path

from utils.moskita_run_report import (
    load_results_dataframe,
    print_training_results_summary,
    summarize_training_results,
)

results_csv = str(Path(globals().get("TRAIN_RESULTS_CSV", Path(run_dir) / "results.csv")))

if os.path.exists(results_csv):
    df = load_results_dataframe(results_csv)
    training_summary = summarize_training_results(df)
    print_training_results_summary(training_summary)
else:
    print(f"⚠️  results.csv not found: {results_csv}")


In [ ]:
# Per-class validation report — run model.val() on the best checkpoint
if not globals().get("RUN_POST_TRAIN_VALIDATION", False):
    print("⏭️  Skipping extra post-train validation to keep the notebook lighter.")
    print("   Set RUN_POST_TRAIN_VALIDATION = True in the Configuration cell to enable it.")
else:
    import os
    from pathlib import Path

    import torch
    import yaml
    from ultralytics import YOLO

    from utils.moskita_dataset_report import summarize_yolo_detection_dataset
    from utils.moskita_run_report import (
        build_detection_report,
        plot_detection_report,
        print_detection_report_summary,
    )

    if "BANNED_CLASS_NAMES" not in globals():
        BANNED_CLASS_NAMES = {"bottle"}
    if "_normalize_class_name" not in globals():
        def _normalize_class_name(name):
            return str(name).strip().lower().replace(" ", "_")

    required_vars = (
        "run_dir",
        "DATA_YAML",
        "IMG_SIZE",
        "CONF_THRESHOLD",
        "IOU_THRESHOLD",
        "BATCH_SIZE",
        "CLASS_NAMES",
    )
    missing_vars = [name for name in required_vars if name not in globals()]
    if missing_vars:
        raise NameError(
            f"Missing required setup variables: {', '.join(missing_vars)}. "
            "Run the setup/configuration cells above first."
        )

    _blocked_class_names = sorted(
        BANNED_CLASS_NAMES.intersection({_normalize_class_name(name) for name in CLASS_NAMES})
    )
    if _blocked_class_names:
        raise ValueError(
            f"Banned classes are present in CLASS_NAMES: {_blocked_class_names}. "
            "Re-run the configuration and dataset assembly cells."
        )

    if "DEVICE" not in globals():
        DEVICE = 0 if torch.cuda.is_available() else "cpu"

    if "splits" not in globals():
        splits = ["train", "val", "test"]

    if "dataset_root" not in globals():
        with open(DATA_YAML, "r") as f:
            _data_cfg = yaml.safe_load(f) or {}

        _yaml_path_val = _data_cfg.get("path", ".")
        if os.path.isabs(_yaml_path_val):
            dataset_root = _yaml_path_val
        else:
            dataset_root = str((Path(DATA_YAML).resolve().parent / _yaml_path_val).resolve())

    _bw = os.path.join(str(run_dir), "weights", "best.pt")
    if not os.path.exists(_bw):
        _bw = os.path.join(str(run_dir), "weights", "last.pt")

    if os.path.exists(_bw):
        print("🔬 Running per-class validation on best weights...")
        _eval_model = YOLO(_bw)

        _model_names_raw = getattr(_eval_model, "names", {})
        if isinstance(_model_names_raw, dict):
            _model_names = [str(_model_names_raw[idx]) for idx in sorted(_model_names_raw)]
        elif isinstance(_model_names_raw, list):
            _model_names = [str(name) for name in _model_names_raw]
        else:
            _model_names = []

        _blocked_model_classes = sorted(
            BANNED_CLASS_NAMES.intersection({_normalize_class_name(name) for name in _model_names})
        )
        if _blocked_model_classes:
            raise ValueError(
                f"Checkpoint '{_bw}' still contains banned classes: {_blocked_model_classes}. "
                "Use bottle-free weights for post-train evaluation."
            )

        _val_kwargs = {
            "data": DATA_YAML,
            "imgsz": IMG_SIZE,
            "conf": CONF_THRESHOLD,
            "iou": IOU_THRESHOLD,
            "verbose": False,
            "batch": max(1, min(8, BATCH_SIZE)),
            "device": DEVICE,
            "workers": 0,
        }

        try:
            _val_results = _eval_model.val(**_val_kwargs)
        except torch.OutOfMemoryError:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print("⚠️  Validation hit a GPU memory limit — retrying on CPU with batch=1.")
            _val_results = _eval_model.val(
                data=DATA_YAML,
                imgsz=IMG_SIZE,
                conf=CONF_THRESHOLD,
                iou=IOU_THRESHOLD,
                verbose=False,
                batch=1,
                device="cpu",
                workers=0,
            )

        if "dataset_summary" not in globals():
            dataset_summary = summarize_yolo_detection_dataset(
                dataset_root,
                CLASS_NAMES,
                splits=splits,
            )

        split_class_counts = dataset_summary.get("split_class_counts", {})
        class_counts = dataset_summary.get("class_counts", {})
        _val_support = split_class_counts.get("val", class_counts)

        if hasattr(_val_results, "box") and _val_results.box is not None:
            detection_report = build_detection_report(
                _val_results.box,
                CLASS_NAMES,
                support_counts=_val_support,
                confusion_matrix=getattr(_val_results, "confusion_matrix", None),
                top_k=10,
            )
            print_detection_report_summary(detection_report)
            plot_detection_report(detection_report)
        else:
            print("⚠️  Val results did not expose .box metrics — check Ultralytics version.")

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    else:
        print("⚠️  No weights found in run directory — run training first.")


In [ ]:
# Training curves with CSV fallback when results.png is unavailable
results_png = os.path.join(run_dir, "results.png")
results_csv = os.path.join(run_dir, "results.csv")

if os.path.exists(results_png):
    img = mpimg.imread(results_png)
    fig, ax = plt.subplots(figsize=(18, 10))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title("Training Curves", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
    plt.close(fig)
elif os.path.exists(results_csv):
    print("⚠️  results.png not found — building the dashboard from results.csv instead.")
    import pandas as pd

    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    if "epoch" in df.columns:
        epoch_numbers = df["epoch"].astype(int)
        display_epochs = epoch_numbers + 1 if epoch_numbers.min() == 0 else epoch_numbers
    else:
        display_epochs = pd.Series(range(1, len(df) + 1))

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    def plot_columns(ax, title, ylabel, column_specs):
        plotted = False
        for label, column, color in column_specs:
            if column in df.columns:
                ax.plot(display_epochs, df[column], label=label, color=color, linewidth=2)
                plotted = True
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.grid(alpha=0.25)
        if plotted:
            ax.legend(fontsize=9)
        else:
            ax.text(0.5, 0.5, "Columns unavailable", ha="center", va="center", transform=ax.transAxes)
            ax.set_xticks([])
            ax.set_yticks([])

    plot_columns(
        axes[0, 0],
        "Validation Accuracy",
        "Score",
        [
            ("mAP@50", "metrics/mAP50(B)", "#4C72B0"),
            ("mAP@50-95", "metrics/mAP50-95(B)", "#DD8452"),
        ],
    )
    plot_columns(
        axes[0, 1],
        "Precision / Recall",
        "Score",
        [
            ("Precision", "metrics/precision(B)", "#55A868"),
            ("Recall", "metrics/recall(B)", "#C44E52"),
        ],
    )
    plot_columns(
        axes[1, 0],
        "Box Loss",
        "Loss",
        [
            ("Train", "train/box_loss", "#4C72B0"),
            ("Val", "val/box_loss", "#DD8452"),
        ],
    )
    plot_columns(
        axes[1, 1],
        "Class / DFL Loss",
        "Loss",
        [
            ("Train cls", "train/cls_loss", "#55A868"),
            ("Val cls", "val/cls_loss", "#C44E52"),
            ("Train dfl", "train/dfl_loss", "#8172B3"),
            ("Val dfl", "val/dfl_loss", "#937860"),
        ],
    )

    plt.suptitle("MosKita — Training Dashboard (from results.csv)", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
    plt.close(fig)
else:
    print("⚠️  Neither results.png nor results.csv was found in the run directory.")


In [ ]:
# Confusion matrix (render each plot as its own figure)
if not globals().get("SHOW_DETAILED_EVAL_PLOTS", False):
    print("⏭️  Skipping separate confusion-matrix figures.")
    print("   The lighter grid/summary cells below are still available.")
else:
    from pathlib import Path

    def _resolve_plot_file(filename_options, search_dirs):
        for directory in search_dirs:
            for filename in filename_options:
                candidate = Path(directory) / filename
                if candidate.exists():
                    return candidate
        return None

    _search_dirs = globals().get("EVAL_PLOT_DIRS")
    if not _search_dirs:
        _search_dirs = [Path(run_dir)]

    plot_specs = [
        ("Confusion Matrix", ["confusion_matrix.png"]),
        ("Confusion Matrix (Normalized)", ["confusion_matrix_normalized.png"]),
    ]

    plot_files = []
    for title, filenames in plot_specs:
        path = _resolve_plot_file(filenames, _search_dirs)
        if path is not None:
            plot_files.append((title, path))

    if plot_files:
        print("🖼️  Rendering confusion matrix plots...")
        for title, path in plot_files:
            fig, ax = plt.subplots(figsize=(10, 10))
            img = mpimg.imread(path)
            ax.imshow(img)
            ax.axis("off")
            ax.set_title(f"{title}\n{path}", fontsize=12, fontweight="bold")
            plt.tight_layout()
            plt.show()
            plt.close(fig)
    else:
        print("⚠️  Confusion matrix plots not found in the expected directories:")
        for directory in _search_dirs:
            print(f"   - {directory}")


In [ ]:
# Precision-Recall and F1 curves (render each plot as its own figure)
if not globals().get("SHOW_DETAILED_EVAL_PLOTS", False):
    print("⏭️  Skipping separate PR/F1/P/R figures.")
    print("   The lighter grid view below is still available.")
else:
    from pathlib import Path

    def _resolve_plot_file(filename_options, search_dirs):
        for directory in search_dirs:
            for filename in filename_options:
                candidate = Path(directory) / filename
                if candidate.exists():
                    return candidate
        return None

    _search_dirs = globals().get("EVAL_PLOT_DIRS")
    if not _search_dirs:
        _search_dirs = [Path(run_dir)]

    pr_plot_specs = [
        ("Precision-Recall Curve", ["BoxPR_curve.png", "PR_curve.png"]),
        ("F1 Score Curve", ["BoxF1_curve.png", "F1_curve.png"]),
        ("Precision Curve", ["BoxP_curve.png", "P_curve.png"]),
        ("Recall Curve", ["BoxR_curve.png", "R_curve.png"]),
    ]

    available = []
    for title, filenames in pr_plot_specs:
        path = _resolve_plot_file(filenames, _search_dirs)
        if path is not None:
            available.append((title, path))

    if available:
        print("🖼️  Rendering PR/F1/P/R plots as separate images...")
        for title, path in available:
            fig, ax = plt.subplots(figsize=(10, 8))
            img = mpimg.imread(path)
            ax.imshow(img)
            ax.axis("off")
            ax.set_title(f"{title}\n{path}", fontsize=12, fontweight="bold")
            plt.tight_layout()
            plt.show()
            plt.close(fig)
    else:
        print("⚠️  PR / F1 curve plots not found in the expected directories.")
        for directory in _search_dirs:
            print(f"   - {directory}")


In [ ]:
# Validation batch predictions (render each image separately)
if not globals().get("SHOW_DETAILED_EVAL_PLOTS", False):
    print("⏭️  Skipping separate validation prediction images.")
    print("   The lighter grid view below is still available.")
else:
    import re
    from pathlib import Path

    from PIL import ImageFile

    ImageFile.LOAD_TRUNCATED_IMAGES = True
    pattern = re.compile(r"^val_batch(\d+)_(labels|pred)\.jpg$")

    _search_dirs = globals().get("EVAL_PLOT_DIRS")
    if not _search_dirs:
        _search_dirs = [Path(run_dir)]

    resolved = {}
    for directory in _search_dirs:
        for path in sorted(Path(directory).glob("val_batch*_*.jpg")):
            if pattern.match(path.name):
                resolved.setdefault(path.name, path)

    max_panels = max(1, int(globals().get("MAX_NOTEBOOK_IMAGE_PANELS", 6)))
    available_val = sorted(
        resolved.items(),
        key=lambda item: (
            int(pattern.match(item[0]).group(1)),
            0 if pattern.match(item[0]).group(2) == "labels" else 1,
        ),
    )[:max_panels]

    if available_val:
        print("🖼️  Rendering validation prediction images separately...")
        for filename, path in available_val:
            match = pattern.match(filename)
            batch_idx = int(match.group(1))
            kind = "Labels" if match.group(2) == "labels" else "Predictions"
            title = f"Val Batch {batch_idx} — {kind}"

            try:
                img = mpimg.imread(path)
            except Exception as exc:
                print(f"⚠️  Skipping unreadable image: {path} ({exc})")
                continue

            fig, ax = plt.subplots(figsize=(12, 10))
            ax.imshow(img)
            ax.axis("off")
            ax.set_title(f"{title}\n{path}", fontsize=12, fontweight="bold")
            plt.tight_layout()
            plt.show()
            plt.close(fig)
    else:
        print("⚠️  Validation prediction images not found in the expected directories.")
        for directory in _search_dirs:
            print(f"   - {directory}")


In [ ]:
# Precision-Recall and F1 curves (grid view)
from pathlib import Path


def _resolve_plot_file(filename_options, search_dirs):
    for directory in search_dirs:
        for filename in filename_options:
            candidate = Path(directory) / filename
            if candidate.exists():
                return candidate
    return None


_search_dirs = globals().get("EVAL_PLOT_DIRS")
if not _search_dirs:
    _search_dirs = [Path(run_dir)]

pr_plot_specs = [
    ("Precision-Recall Curve", ["BoxPR_curve.png", "PR_curve.png"]),
    ("F1 Score Curve", ["BoxF1_curve.png", "F1_curve.png"]),
    ("Precision Curve", ["BoxP_curve.png", "P_curve.png"]),
    ("Recall Curve", ["BoxR_curve.png", "R_curve.png"]),
]

available = []
for title, filenames in pr_plot_specs:
    path = _resolve_plot_file(filenames, _search_dirs)
    if path is not None:
        available.append((title, path))

if available:
    cols = min(2, len(available))
    rows = (len(available) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(10 * cols, 8 * rows))
    axes = np.array(axes).flatten() if len(available) > 1 else [axes]

    for idx, (title, path) in enumerate(available):
        img = mpimg.imread(path)
        axes[idx].imshow(img)
        axes[idx].axis("off")
        axes[idx].set_title(f"{title}\n{path}", fontsize=12, fontweight="bold")

    for idx in range(len(available), len(axes)):
        axes[idx].axis("off")

    plt.tight_layout()
    plt.show()
    plt.close(fig)
else:
    print("⚠️  PR / F1 curve plots not found in the expected directories.")
    for directory in _search_dirs:
        print(f"   - {directory}")


In [ ]:
# Validation batch predictions (visual check, grid view)
import re
from pathlib import Path

pattern = re.compile(r"^val_batch(\d+)_(labels|pred)\.jpg$")

_search_dirs = globals().get("EVAL_PLOT_DIRS")
if not _search_dirs:
    _search_dirs = [Path(run_dir)]

resolved = {}
for directory in _search_dirs:
    for path in sorted(Path(directory).glob("val_batch*_*.jpg")):
        if pattern.match(path.name):
            resolved.setdefault(path.name, path)

max_panels = max(1, int(globals().get("MAX_NOTEBOOK_IMAGE_PANELS", 6)))
available_val = sorted(
    resolved.items(),
    key=lambda item: (
        int(pattern.match(item[0]).group(1)),
        0 if pattern.match(item[0]).group(2) == "labels" else 1,
    ),
)[:max_panels]

if available_val:
    cols = min(2, len(available_val))
    rows = (len(available_val) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(10 * cols, 6 * rows))
    axes = np.array(axes).flatten() if len(available_val) > 1 else [axes]

    for idx, (filename, path) in enumerate(available_val):
        match = pattern.match(filename)
        batch_idx = int(match.group(1))
        kind = "Labels" if match.group(2) == "labels" else "Predictions"
        title = f"Val Batch {batch_idx} — {kind}"

        img = mpimg.imread(path)
        axes[idx].imshow(img)
        axes[idx].axis("off")
        axes[idx].set_title(f"{title}\n{path}", fontsize=11, fontweight="bold")

    for idx in range(len(available_val), len(axes)):
        axes[idx].axis("off")

    plt.tight_layout()
    plt.show()
    plt.close(fig)
else:
    print("⚠️  Validation prediction images not found in the expected directories.")
    for directory in _search_dirs:
        print(f"   - {directory}")


---
### 7 · Model Export (ONNX / TFLite)

Export the best weights for Raspberry Pi 5 edge deployment.
ONNX export should work in this environment. TFLite export is optional and requires a compatible TensorFlow + protobuf stack in the notebook kernel.


In [ ]:
# Load best weights from the training run
from pathlib import Path

from utils.moskita_run_report import collect_saved_model_paths

_saved_paths = collect_saved_model_paths(globals().get("TRAIN_RUN_DIR", run_dir))
best_weights = str(_saved_paths["best_weights"])
last_weights = str(_saved_paths["last_weights"])

if Path(best_weights).exists():
    print(f"📦 Loading best weights: {best_weights}")
    best_model = YOLO(best_weights)
    print(f"✅ Best model loaded successfully from: {best_weights}")
elif Path(last_weights).exists():
    print("⚠️  best.pt not found — using last trained model instead.")
    best_model = YOLO(last_weights)
    print(f"✅ Last model loaded: {last_weights}")
else:
    best_model = model
    print("⚠️  No saved weights found — using current model in memory.")


In [ ]:
import os
import re
import shutil
from pathlib import Path

# Build deployment-friendly filenames that include model specs + trained epochs.
results_csv = os.path.join(run_dir, "results.csv")
trained_epochs = int(EPOCHS)

if os.path.exists(results_csv):
    import pandas as pd

    _train_df = pd.read_csv(results_csv)
    _train_df.columns = _train_df.columns.str.strip()
    if "epoch" in _train_df.columns and len(_train_df) > 0:
        _epoch_series = _train_df["epoch"].astype(int)
        trained_epochs = int(_epoch_series.max())
        if int(_epoch_series.min()) == 0:
            trained_epochs += 1

TRAINED_EPOCHS = trained_epochs
WEBAPP_MODEL_DIR = str((REPO_ROOT / "deploy" / "webapp" / "public" / "models").resolve())

_run_slug = re.sub(r"[^a-zA-Z0-9]+", "-", RUN_NAME).strip("-").lower()
_variant_slug = re.sub(r"[^a-zA-Z0-9]+", "-", Path(MODEL_VARIANT).stem).strip("-").lower()
MODEL_EXPORT_STEM = f"moskita_{_run_slug}_{_variant_slug}_img{IMG_SIZE}_ep{TRAINED_EPOCHS}"
ONNX_EXPORT_FILENAME = f"{MODEL_EXPORT_STEM}.onnx"
TFLITE_EXPORT_FILENAME = f"{MODEL_EXPORT_STEM}.tflite"

print("🚀 Exporting to ONNX...")
print(f"   Export stem   : {MODEL_EXPORT_STEM}")
onnx_path = best_model.export(
    format="onnx",
    imgsz=IMG_SIZE,
    simplify=True,
    dynamic=True,        # enables variable batch size at inference time
)
print(f"✅ ONNX exported: {onnx_path}")

export_dir = Path(EXPORT_DIR).resolve()
webapp_dir = Path(WEBAPP_MODEL_DIR).resolve()
export_dir.mkdir(parents=True, exist_ok=True)
webapp_dir.mkdir(parents=True, exist_ok=True)

onnx_dest_export = export_dir / ONNX_EXPORT_FILENAME
onnx_dest_webapp = webapp_dir / ONNX_EXPORT_FILENAME
shutil.copy2(onnx_path, onnx_dest_export)
shutil.copy2(onnx_path, onnx_dest_webapp)

# Keep a stable alias so the webapp default path keeps working.
onnx_alias_export = export_dir / "moskita.onnx"
onnx_alias_webapp = webapp_dir / "moskita.onnx"
shutil.copy2(onnx_path, onnx_alias_export)
shutil.copy2(onnx_path, onnx_alias_webapp)

print(f"✅ Saved spec file   : {onnx_dest_export}")
print(f"✅ Webapp spec file  : {onnx_dest_webapp}")
print(f"✅ Webapp default    : {onnx_alias_webapp}")

In [ ]:
# Post-training sample inference runs (bottle-filtered visualizations)
# Prefer test split and include a balanced mix of available images.
if not globals().get("SHOW_DETAILED_INFERENCE_PLOTS", False):
    print("⏭️  Skipping separate per-image inference figures.")
    print("   The lighter grid view below is still available.")
else:
    import matplotlib.patches as patches

    if "BANNED_CLASS_NAMES" not in globals():
        BANNED_CLASS_NAMES = {"bottle"}
    if "_normalize_class_name" not in globals():
        def _normalize_class_name(name):
            return str(name).strip().lower().replace(" ", "_")

    test_img_dir = os.path.join(dataset_root, "test", "images")
    val_img_dir = os.path.join(dataset_root, "val", "images")

    def _clean_sample_name(filename):
        name = os.path.basename(filename)
        stem, _ = os.path.splitext(name)
        if ".rf." in stem:
            stem = stem.split(".rf.", 1)[0]

        strip_prefixes = [
            "k1taru_local_train_",
            "k1taru_local_valid_",
            "k1taru_local_test_",
            "adnans_breeding_train_",
            "adnans_breeding_valid_",
            "adnans_breeding_test_",
            "roboflow_public_train_",
            "roboflow_public_valid_",
            "roboflow_public_test_",
            "k1taru_",
            "adnans_",
            "roboflow_",
        ]
        for prefix in strip_prefixes:
            if stem.startswith(prefix):
                stem = stem[len(prefix):]
                break

        return stem.replace("_", " ")

    def _collect_images(image_dir):
        if not os.path.exists(image_dir):
            return []
        return [
            os.path.join(image_dir, f)
            for f in os.listdir(image_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ]

    def _balanced_paths(paths, total=6):
        total = min(total, len(paths))
        if total <= 0:
            return []

        own_paths = [p for p in paths if os.path.basename(p).lower().startswith("k1taru_")]
        other_paths = [p for p in paths if p not in own_paths]

        selected = []
        if own_paths and other_paths:
            own_quota = min(len(own_paths), max(1, total // 3))
            other_quota = min(len(other_paths), total - own_quota)
            selected.extend(random.sample(own_paths, own_quota))
            selected.extend(random.sample(other_paths, other_quota))
        else:
            selected.extend(random.sample(paths, total))

        if len(selected) < total:
            chosen = set(selected)
            remaining = [p for p in paths if p not in chosen]
            if remaining:
                selected.extend(random.sample(remaining, min(total - len(selected), len(remaining))))

        random.shuffle(selected)
        return selected

    def _resolve_pred_class_name(result, cls_id):
        names = getattr(result, "names", None)
        if isinstance(names, dict) and cls_id in names:
            return str(names[cls_id])
        if isinstance(names, list) and 0 <= cls_id < len(names):
            return str(names[cls_id])
        if 0 <= cls_id < len(CLASS_NAMES):
            return str(CLASS_NAMES[cls_id])
        return str(cls_id)

    def _render_filtered_boxes(ax, result):
        kept_rows = []
        dropped_count = 0

        orig_img = getattr(result, "orig_img", None)
        if orig_img is None:
            annotated = result.plot()
            ax.imshow(annotated[:, :, ::-1])
            ax.axis("off")
            return kept_rows, dropped_count

        ax.imshow(orig_img[:, :, ::-1])
        ax.axis("off")

        boxes = getattr(result, "boxes", None)
        if boxes is None or len(boxes) == 0:
            return kept_rows, dropped_count

        for box in boxes:
            cls_id = int(box.cls.item())
            cls_name = _resolve_pred_class_name(result, cls_id)
            if _normalize_class_name(cls_name) in BANNED_CLASS_NAMES:
                dropped_count += 1
                continue

            conf = float(box.conf.item())
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            color = plt.cm.tab20(cls_id % 20)

            rect = patches.Rectangle(
                (x1, y1),
                max(1.0, x2 - x1),
                max(1.0, y2 - y1),
                linewidth=2,
                edgecolor=color,
                facecolor="none",
            )
            ax.add_patch(rect)
            ax.text(
                x1,
                max(0.0, y1 - 4.0),
                f"{cls_name} {conf:.2f}",
                color="white",
                fontsize=8,
                bbox=dict(facecolor=color, alpha=0.85, pad=1.5),
            )
            kept_rows.append((cls_name, conf))

        return kept_rows, dropped_count

    candidate_paths = _collect_images(test_img_dir)
    max_panels = max(1, int(globals().get("MAX_NOTEBOOK_IMAGE_PANELS", 6)))
    if len(candidate_paths) < max_panels:
        candidate_paths.extend(_collect_images(val_img_dir))
    if len(candidate_paths) < max_panels:
        candidate_paths.extend(_collect_images(train_img_dir))

    selected_paths = _balanced_paths(candidate_paths, total=max_panels)

    inference_model = globals().get("best_model")
    if inference_model is None:
        inference_model = globals().get("model")
    if inference_model is None:
        from ultralytics import YOLO

        _best_weights = os.path.join(run_dir, "weights", "best.pt")
        _last_weights = os.path.join(run_dir, "weights", "last.pt")
        if os.path.exists(_best_weights):
            inference_model = YOLO(_best_weights)
        elif os.path.exists(_last_weights):
            inference_model = YOLO(_last_weights)

    if selected_paths and inference_model is not None:
        print(f"🔎 Running inference on {len(selected_paths)} sample images")
        print()

        detection_rows = []
        dropped_total = 0

        for sample_path in selected_paths:
            inf_results = inference_model(sample_path, conf=CONF_THRESHOLD, iou=IOU_THRESHOLD, verbose=False)
            if not inf_results:
                continue

            r = inf_results[0]
            image_label = _clean_sample_name(os.path.basename(sample_path))

            fig, ax = plt.subplots(figsize=(8, 6))
            kept_rows, dropped_count = _render_filtered_boxes(ax, r)
            kept_count = len(kept_rows)
            dropped_total += dropped_count

            suffix = f", filtered {dropped_count}" if dropped_count else ""
            ax.set_title(f"{image_label[:60]} ({kept_count}{suffix})", fontsize=11, fontweight="bold")
            plt.tight_layout()
            plt.show()
            plt.close(fig)

            for cls_name, conf in kept_rows:
                detection_rows.append((image_label, cls_name, conf))

        if dropped_total:
            print(f"⚠️  Filtered out {dropped_total} banned-class detections: {sorted(BANNED_CLASS_NAMES)}")

        if detection_rows:
            print("📋 Detection details")
            print(f"{'Image':<30} {'Class':<30} {'Confidence':>10}")
            print("-" * 74)
            for image_label, cls_name, conf in detection_rows:
                print(f"{image_label[:30]:<30} {cls_name:<30} {conf:>10.4f}")
        else:
            print("No allowed-class detections found in the selected sample images.")
    elif inference_model is None:
        print("⚠️  No model available for inference.")
        print("   Run the training/load-best-model cells first.")
    else:
        print("⚠️  No sample images available for inference test.")
        print("   Add images to your dataset first, then re-run this cell.")


---
### 8 · Quick Inference Test

Run a quick inference on a sample image to verify the trained model works end-to-end.


In [ ]:
# Post-training sample inference runs (bottle-filtered grid view)
# Prefer test split and include a balanced mix of available images.
import matplotlib.patches as patches

if "BANNED_CLASS_NAMES" not in globals():
    BANNED_CLASS_NAMES = {"bottle"}
if "_normalize_class_name" not in globals():
    def _normalize_class_name(name):
        return str(name).strip().lower().replace(" ", "_")

test_img_dir = os.path.join(dataset_root, "test", "images")
val_img_dir = os.path.join(dataset_root, "val", "images")


def _clean_sample_name(filename):
    name = os.path.basename(filename)
    stem, _ = os.path.splitext(name)
    if ".rf." in stem:
        stem = stem.split(".rf.", 1)[0]

    strip_prefixes = [
        "k1taru_local_train_",
        "k1taru_local_valid_",
        "k1taru_local_test_",
        "adnans_breeding_train_",
        "adnans_breeding_valid_",
        "adnans_breeding_test_",
        "roboflow_public_train_",
        "roboflow_public_valid_",
        "roboflow_public_test_",
        "k1taru_",
        "adnans_",
        "roboflow_",
    ]
    for prefix in strip_prefixes:
        if stem.startswith(prefix):
            stem = stem[len(prefix):]
            break

    return stem.replace("_", " ")


def _collect_images(image_dir):
    if not os.path.exists(image_dir):
        return []
    return [
        os.path.join(image_dir, f)
        for f in os.listdir(image_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]


def _balanced_paths(paths, total=6):
    total = min(total, len(paths))
    if total <= 0:
        return []

    own_paths = [p for p in paths if os.path.basename(p).lower().startswith("k1taru_")]
    other_paths = [p for p in paths if p not in own_paths]

    selected = []
    if own_paths and other_paths:
        own_quota = min(len(own_paths), max(1, total // 3))
        other_quota = min(len(other_paths), total - own_quota)
        selected.extend(random.sample(own_paths, own_quota))
        selected.extend(random.sample(other_paths, other_quota))
    else:
        selected.extend(random.sample(paths, total))

    if len(selected) < total:
        chosen = set(selected)
        remaining = [p for p in paths if p not in chosen]
        if remaining:
            selected.extend(random.sample(remaining, min(total - len(selected), len(remaining))))

    random.shuffle(selected)
    return selected


def _resolve_pred_class_name(result, cls_id):
    names = getattr(result, "names", None)
    if isinstance(names, dict) and cls_id in names:
        return str(names[cls_id])
    if isinstance(names, list) and 0 <= cls_id < len(names):
        return str(names[cls_id])
    if 0 <= cls_id < len(CLASS_NAMES):
        return str(CLASS_NAMES[cls_id])
    return str(cls_id)


def _render_filtered_boxes(ax, result):
    kept_rows = []
    dropped_count = 0

    orig_img = getattr(result, "orig_img", None)
    if orig_img is None:
        annotated = result.plot()
        ax.imshow(annotated[:, :, ::-1])
        ax.axis("off")
        return kept_rows, dropped_count

    ax.imshow(orig_img[:, :, ::-1])
    ax.axis("off")

    boxes = getattr(result, "boxes", None)
    if boxes is None or len(boxes) == 0:
        return kept_rows, dropped_count

    for box in boxes:
        cls_id = int(box.cls.item())
        cls_name = _resolve_pred_class_name(result, cls_id)
        if _normalize_class_name(cls_name) in BANNED_CLASS_NAMES:
            dropped_count += 1
            continue

        conf = float(box.conf.item())
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        color = plt.cm.tab20(cls_id % 20)

        rect = patches.Rectangle(
            (x1, y1),
            max(1.0, x2 - x1),
            max(1.0, y2 - y1),
            linewidth=2,
            edgecolor=color,
            facecolor="none",
        )
        ax.add_patch(rect)
        ax.text(
            x1,
            max(0.0, y1 - 4.0),
            f"{cls_name} {conf:.2f}",
            color="white",
            fontsize=8,
            bbox=dict(facecolor=color, alpha=0.85, pad=1.5),
        )
        kept_rows.append((cls_name, conf))

    return kept_rows, dropped_count


candidate_paths = _collect_images(test_img_dir)
max_panels = max(1, int(globals().get("MAX_NOTEBOOK_IMAGE_PANELS", 6)))
if len(candidate_paths) < max_panels:
    candidate_paths.extend(_collect_images(val_img_dir))
if len(candidate_paths) < max_panels:
    candidate_paths.extend(_collect_images(train_img_dir))

selected_paths = _balanced_paths(candidate_paths, total=max_panels)

inference_model = globals().get("best_model")
if inference_model is None:
    inference_model = globals().get("model")
if inference_model is None:
    from ultralytics import YOLO

    _best_weights = os.path.join(run_dir, "weights", "best.pt")
    _last_weights = os.path.join(run_dir, "weights", "last.pt")
    if os.path.exists(_best_weights):
        inference_model = YOLO(_best_weights)
    elif os.path.exists(_last_weights):
        inference_model = YOLO(_last_weights)

if selected_paths and inference_model is not None:
    print(f"🔎 Running inference on {len(selected_paths)} sample images")
    print()

    cols = min(3, len(selected_paths))
    rows = (len(selected_paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4.8 * rows))
    axes = np.array(axes).flatten() if len(selected_paths) > 1 else [axes]

    detection_rows = []
    dropped_total = 0

    for idx, sample_path in enumerate(selected_paths):
        inf_results = inference_model(
            sample_path,
            conf=CONF_THRESHOLD,
            iou=IOU_THRESHOLD,
            verbose=False,
        )
        if not inf_results:
            axes[idx].axis("off")
            continue

        r = inf_results[0]
        image_label = _clean_sample_name(os.path.basename(sample_path))

        kept_rows, dropped_count = _render_filtered_boxes(axes[idx], r)
        dropped_total += dropped_count
        kept_count = len(kept_rows)
        suffix = f", filtered {dropped_count}" if dropped_count else ""
        axes[idx].set_title(f"{image_label[:30]} ({kept_count}{suffix})", fontsize=9)

        for cls_name, conf in kept_rows:
            detection_rows.append((image_label, cls_name, conf))

    for idx in range(len(selected_paths), len(axes)):
        axes[idx].axis("off")

    plt.suptitle("MosKita — Post-Training Sample Runs (Bottle Filtered)", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
    plt.close(fig)

    if dropped_total:
        print(f"⚠️  Filtered out {dropped_total} banned-class detections: {sorted(BANNED_CLASS_NAMES)}")

    if detection_rows:
        print("📋 Detection details")
        print(f"{'Image':<30} {'Class':<30} {'Confidence':>10}")
        print("-" * 74)
        for image_label, cls_name, conf in detection_rows:
            print(f"{image_label[:30]:<30} {cls_name:<30} {conf:>10.4f}")
    else:
        print("No allowed-class detections found in the selected sample images.")
elif inference_model is None:
    print("⚠️  No model available for inference.")
    print("   Run the training/load-best-model cells first.")
else:
    print("⚠️  No sample images available for inference test.")
    print("   Add images to your dataset first, then re-run this cell.")


In [ ]:
# Training artifact gallery (render each artifact as its own figure)
if not globals().get("SHOW_TRAINING_ARTIFACT_GALLERY", False):
    print("⏭️  Skipping training artifact gallery to keep the notebook output light.")
    print("   Set SHOW_TRAINING_ARTIFACT_GALLERY = True in the Configuration cell to enable it.")
else:
    import re
    from pathlib import Path

    def _dedupe_existing_dirs(paths):
        resolved = []
        seen = set()
        for path in paths:
            p = Path(path)
            if not p.exists() or not p.is_dir():
                continue
            key = str(p.resolve())
            if key in seen:
                continue
            seen.add(key)
            resolved.append(p.resolve())
        return resolved

    artifact_dirs = [Path(run_dir)]
    run_parent = Path(run_dir).parent if "run_dir" in globals() else None
    if run_parent is not None and run_parent.exists() and "RUN_NAME" in globals():
        sibling_runs = sorted(
            run_parent.glob(f"{RUN_NAME}*"),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
        artifact_dirs.extend(sibling_runs)

    artifact_dirs = _dedupe_existing_dirs(artifact_dirs)

    resolved = {}
    for directory in artifact_dirs:
        label_path = directory / "labels.jpg"
        if label_path.exists():
            resolved.setdefault("labels.jpg", label_path)

        for path in sorted(directory.glob("train_batch*.jpg")):
            resolved.setdefault(path.name, path)

    batch_pattern = re.compile(r"^train_batch(\d+)\.jpg$")
    max_panels = max(1, int(globals().get("MAX_NOTEBOOK_IMAGE_PANELS", 6)))
    available_artifacts = sorted(
        resolved.items(),
        key=lambda item: (
            0 if item[0] == "labels.jpg" else 1,
            int(batch_pattern.match(item[0]).group(1)) if batch_pattern.match(item[0]) else 10**9,
        ),
    )[:max_panels]

    if available_artifacts:
        print("🖼️  Rendering training artifacts as separate images...")
        for filename, path in available_artifacts:
            if filename == "labels.jpg":
                title = "Label Sanity Check"
            else:
                match = batch_pattern.match(filename)
                batch_idx = int(match.group(1)) if match else "?"
                title = f"Train Batch {batch_idx}"

            fig, ax = plt.subplots(figsize=(12, 5))
            img = mpimg.imread(path)
            ax.imshow(img)
            ax.axis("off")
            ax.set_title(f"{title}\n{path}", fontsize=12, fontweight="bold")
            plt.tight_layout()
            plt.show()
            plt.close(fig)
    else:
        print("⚠️  No training artifact images found in the run directory.")
        print(f"   Checked: {run_dir}")
